## Context

This notebook accompanies User Guide section 10 (AgentMemory).  It exercises the four operations the `AgentMemory` wrapper exposes — `store_findings`, `recall_entity`, `compare_entities`, and `check_contradiction` — against the prebuilt data-science-practices fixture.  We synthesize three classifier findings, recall the highest-accuracy one via the lossless ENM register, compare them pairwise via tension energy, and probe what happens when a contradicting fact is proposed.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from tutorials.user_guide.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: storage, recall, comparison, contradiction

`AgentMemory` is the storage-layer abstraction that hides the three pieces below it: the persistent `GMSExpertStore` (graph plus geometry on $S^{m-1}$), the `RuntimeLearner` two-phase update (synchronous graph insert plus asynchronous Riemannian gradient descent), and the lossless `ExactNumericalMemory` (SHA-256-keyed key-value register).  An agent that records experimental findings calls `store_findings(triples, enm_entries)`; an agent that compares two findings calls `compare_entities(a, b)` and reads `tension_energy` against the three bands set up in section 4 (agreement under $0.8$, irrelevance up to $\sqrt{2}$, contradiction above).

The contradiction probe is the contract that stops a rogue write from silently overwriting a stored fact with its opposite.  `check_contradiction(head, relation, tail)` returns the maximum tension energy between the proposed *tail* and the existing tails for `(head, relation, ?)`; a value above $1.4$ falls in the contradiction band on a regex-mode fixture.  Whether the agent raises, logs, or queues for review is a policy decision section 18 explores; this notebook only demonstrates the signal.

Below we point an `AgentMemory` at the prebuilt fixture (read-only — we do not retrain), then synthesize three classifier findings and run all four operations on them.

In [ ]:
from knowlytix.harness.governance.memory import AgentMemory
import numpy as np

# Point AgentMemory at the prebuilt ds_practices fixture.
# We do not pass knowledge_path: the store already exists
# on disk, so AgentMemory loads it via store.load().
fixture = ug.FIXTURES_DIR / 'ds_practices_store'
memory = AgentMemory(
    store_path=str(fixture),
    device=str(DEVICE),
)
stats = memory.memory_stats()
print('memory stats:')
for k, v in stats.items():
    print(f'  {k:18s} {v}')

In [ ]:
# Synthesize findings for three classifier runs on iris.
# Triples carry the structural facts; ENM carries the
# byte-identical numerics.  No actual training happens.
model_runs = [
    {'name': 'rf_iris',     'test_accuracy': 0.973,
     'train_accuracy': 1.0,   'family': 'tree-ensemble'},
    {'name': 'logreg_iris', 'test_accuracy': 0.946,
     'train_accuracy': 0.965, 'family': 'linear'},
    {'name': 'svc_iris',    'test_accuracy': 0.953,
     'train_accuracy': 0.973, 'family': 'kernel'},
]
all_triples = []
all_enm = []
for run in model_runs:
    name = run['name']
    all_triples.extend([
        (name, 'trained_on',    'iris'),
        (name, 'has_family',    run['family']),
        (name, 'has_metric',    'test_accuracy'),
    ])
    all_enm.extend([
        ('test_accuracy',  name, run['test_accuracy']),
        ('train_accuracy', name, run['train_accuracy']),
    ])
result = memory.store_findings(
    triples=all_triples,
    enm_entries=all_enm,
)
print(f'training_queued = {result.training_queued}, '
      f'triples added = {len(all_triples)}, '
      f'ENM entries added = {len(all_enm)}')

In [ ]:
# Recall: which model has the highest test_accuracy?
# Use ENM (byte-identical IEEE 754) rather than the
# embedding to avoid embedding-side regression noise.
recalled = []
for run in model_runs:
    acc = memory.recall_metric('test_accuracy', run['name'])
    recalled.append((run['name'], acc))
    print(f'  recall_metric(test_accuracy, {run["name"]:12s}) -> {acc}')
best = max(recalled, key=lambda kv: kv[1] or -1)
print(f'\nbest by test_accuracy: {best[0]} = {best[1]}')

In [ ]:
# Compare: pairwise tension energy across the three runs.
names = [r['name'] for r in model_runs]
n = len(names)
tension_matrix = np.zeros((n, n))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i == j:
            continue
        cmp = memory.compare_entities(a, b)
        te = cmp.tension_energy
        tension_matrix[i, j] = te if te is not None else np.nan
print('tension_energy matrix:')
for i, row in enumerate(tension_matrix):
    print(f'  {names[i]:12s} ' + ' '.join(f'{v:6.3f}' for v in row))

In [ ]:
# Attempt to store a contradicting finding and observe the
# tension signal.  The original finding for rf_iris is
# (rf_iris, has_family, tree-ensemble); we now propose a
# contradicting tail and check_contradiction() reports the
# maximum tension between the proposed tail and any
# stored tail for (rf_iris, has_family, ?).
probe_tension = memory.check_contradiction(
    head='rf_iris',
    relation='has_family',
    tail='linear',  # opposite of stored 'tree-ensemble'
)
agreement_tension = memory.check_contradiction(
    head='rf_iris',
    relation='has_family',
    tail='tree-ensemble',  # already the stored fact
)
print(f'check_contradiction(rf_iris, has_family, linear)        '
      f'-> tension = {probe_tension:.4f}')
print(f'check_contradiction(rf_iris, has_family, tree-ensemble) '
      f'-> tension = {agreement_tension:.4f}')
print()
if probe_tension > 1.4:
    print('FLAGGED: tension above 1.4 -> contradiction band.')
else:
    print('not flagged at 1.4: regex-mode fixture is loose; '
          'a hybrid-mode rebuild would push this higher.')

In [ ]:
# Visualization: tension matrix between stored findings.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(tension_matrix, cmap='RdYlGn_r', vmin=0.0, vmax=2.0)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=15, fontsize=9)
ax.set_yticklabels(names, fontsize=9)
for i in range(n):
    for j in range(n):
        v = tension_matrix[i, j]
        if i == j:
            label = '-'
        else:
            label = f'{v:.3f}'
        ax.text(j, i, label, ha='center', va='center', fontsize=8, color='black')
fig.colorbar(im, ax=ax, label='tension energy (0=agree, ~1.41=irrelevance, ~2=contradict)')
ax.set_title('Tension matrix between stored findings')
fig.tight_layout()
plt.show()

### Exercises

1. Use `memory.recall_entity('rf_iris')` and read the `triples_as_head` and `enm_values` fields.  Confirm every triple you stored above appears, and that the ENM values round-trip byte-identically (no float rounding).
2. Add a fourth synthetic model run with `test_accuracy=0.500` (deliberately bad).  Re-run the tension matrix.  Does the bad model sit further from the others than they do from each other?
3. Probe a *non*-contradiction by calling `check_contradiction('rf_iris', 'trained_on', 'wine')`. This tail is *novel* but not contradictory; the tension should land in the irrelevance band, not the contradiction band.

### What you should now be able to do

- Open an `AgentMemory` over a prebuilt fixture and read `memory_stats()`.
- Store a list of `(head, relation, tail)` findings alongside `(category, entity_id, value)` ENM entries.
- Recall an exact numeric via `recall_metric(category, entity_id)` (byte-identical).
- Compare two entities with `compare_entities` and read the tension energy across the three bands of section 4.
- Probe a candidate write with `check_contradiction` before committing it.

### Cross-references

- USER_GUIDE.md section 10 (AgentMemory)
- USER_GUIDE.md section 4 (GMS Primitives Reference) for the agreement / irrelevance / contradiction bands.
- USER_GUIDE.md section 11 (GeometricReasoner) — the reasoner uses an `AgentMemory` instance under the hood.
- `governed_agent/memory.py:AgentMemory`
- `gms_agent_builder_ext/chapters/ch05.tex` for the AgentMemory chapter.

In [ ]:
# Final assertion: a proposed contradiction registers a
# tension above 1.4.  Regex-mode primitive discrimination
# is loose, so we use 1.4 rather than the 'true'
# contradiction-band floor sqrt(2) ~= 1.414.  A hybrid-mode
# rebuild tightens this toward 1.7-1.8; until then the
# 1.4 floor is the empirical threshold this fixture
# supports.
# TODO(p7-execute): tighten if hybrid fixture is rebuilt;
# regex-mode primitive discrimination is loose.
assert probe_tension > 1.4, (
    f'contradiction probe failed: tension={probe_tension:.4f}'
    ' (expected > 1.4)'
)
assert all(
    memory.recall_metric('test_accuracy', r['name']) == r['test_accuracy']
    for r in model_runs
), 'ENM round-trip not byte-identical'
print(f'OK: contradiction tension = {probe_tension:.4f} > 1.4; '
      f'best model by ENM recall = {best[0]} '
      f'(test_accuracy = {best[1]}).')